In [ ]:
#| default_exp format
%load_ext autoreload
%autoreload 2

In [ ]:
#| hide
import sys, os
from pathlib import Path

In [ ]:
#| hide

# Insert in Path Project Directory
sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
# | export
import re
import json
from typing import Tuple
from collections import defaultdict


import pandas as pd
from rich import print as pp
from dotenv import find_dotenv, load_dotenv
from fastcore.utils import listify
from fastcore.foundation import L

from geopy.distance import geodesic
from tqdm.auto import tqdm

from .constants import (
	APP_ANALISE_EN,
	APP_ANALISE_PT,
	BW,
	CAT_COLUMNS,
	FLOAT_COLUMNS,
	INT_COLUMNS,
	RE_BW,
	STR_COLUMNS,
)

MAX_DIST = 10  # Km
LIMIT_FREQ = 84812.50
load_dotenv(find_dotenv(), override=True)
pd.options.mode.copy_on_write = True
tqdm.pandas()

True

# Formatação

> Este módulo possui funções auxiliares de formatação dos dados das várias fontes.


In [ ]:
# | export
def parse_bw(
	bw: str,  # Designação de Emissão (Largura + Classe) codificada como string
) -> Tuple[str, str]:  # Largura e Classe de Emissão
	"""Parse the bandwidth string"""
	if match := re.match(RE_BW, bw):
		multiplier = BW[match[2]]
		if mantissa := match[3]:
			number = float(f'{match[1]}.{mantissa}')
		else:
			number = float(match[1])
		classe = match[4]
		return str(multiplier * number), str(classe)
	return pd.NA, pd.NA

## Mesclagem
Função auxiliar para mesclar registros que são iguais das diversas bases, i.e. estão a uma distância menor que `MAX_DIST` e verificar a validade da mesclagem

In [ ]:
# | export
def get_km_distance(row):
	return geodesic((row.iloc[0], row.iloc[1]), (row.iloc[2], row.iloc[3])).km


def merge_on_frequency(
	df_left: pd.DataFrame,  # Left DataFrame
	df_right: pd.DataFrame,  # Right DataFrame
	on: str = 'Frequência',  # Column to use as merge key
	cols2merge: Tuple = ('Entidade', 'Fonte'),  # Colums to concatenate
) -> pd.DataFrame:  # Resulted merged dataframe
	"""Merge the dataframes based on Frequency
	It's assumed the have the same columns or one set is contained in the other,
	otherwise the various filters won't work as expected
	"""
	df_left = df_left.astype('string', copy=False).drop_duplicates(ignore_index=True)
	df_right = df_right.astype('string', copy=False).drop_duplicates(ignore_index=True)
	df = pd.merge(
		df_left,
		df_right,
		on=on,
		how='outer',
		indicator=True,
		copy=False,
	)

	left_suffix, right_suffix = '_x', '_y'
	lat, long = 'Latitude', 'Longitude'

	left_only = df._merge == 'left_only'
	right_only = df._merge == 'right_only'
	both = df._merge == 'both'
	df = df.drop(columns=['_merge'])

	# Disjuntos
	if df[both].empty:
		return pd.concat([df_left, df_right], ignore_index=True)

	only_left = df[left_only].copy()
	only_left = only_left.iloc[:, : len(df_left.columns)]
	left_cols = df.columns[: len(df_left.columns)].to_list()
	only_left.columns = df_left.columns

	only_right = df[right_only].copy()
	right_cols = listify(on) + df.columns[len(df_left.columns) :].to_list()
	only_right = only_right.loc[:, right_cols]
	only_right.columns = df_right.columns

	intersection_left = len(df_left) - len(only_left)
	intersection_right = len(df_right) - len(only_right)

	both_columns = [
		f'{lat}{left_suffix}',
		f'{long}{left_suffix}',
		f'{lat}{right_suffix}',
		f'{long}{right_suffix}',
	]

	pp('[bold green]Logging:[/bold green] [italic]Calculando distância entre estações mescladas.')
	df.loc[both, 'Distance'] = df.loc[both, both_columns].progress_apply(get_km_distance, axis=1)

	df_both = df[both].sort_values('Distance', ignore_index=True)

	filter_left_cols = df_both.columns[: len(df_left.columns)].to_list()
	filter_right_cols = (
		listify(on) + df_both.columns[len(df_left.columns) : -1].to_list()
	)  # the -1 is to eliminate the distance

	# keep only the closer merged rows in the outer join
	df_both_left = df_both.copy().drop_duplicates(filter_left_cols, keep='first', ignore_index=True)
	df_both_right = df_both.copy().drop_duplicates(
		filter_right_cols, keep='first', ignore_index=True
	)

	# Sanity Checks
	assert (
		len(df_both_left) == intersection_left
	), f'Grouping by unique columns has unexpected length: {len(df_both_left)}!= {intersection_left}'

	assert (
		len(df_both_right) == intersection_right
	), f'Grouping by unique columns has unexpected length: {len(df_both_right)}!= {intersection_right}'

	# Separate according the MAX_DIST
	df_both_far_left = df_both_left[df_both_left.Distance > MAX_DIST]
	df_both_far_right = df_both_right[df_both_right.Distance > MAX_DIST]

	df_both_left = df_both_left[df_both_left.Distance <= MAX_DIST]
	df_both_right = df_both_right[df_both_right.Distance <= MAX_DIST]

	merge_cols = df_both.columns.to_list()
	merge_cols.remove('Distance')
	# Since it's an outer join, keep only the columns that are in both
	df_close_merge = (
		pd.merge(df_both_left, df_both_right, how='inner', on=merge_cols, copy=False)
		.drop('Distance_y', axis=1)
		.rename(columns={'Distance_x': 'Distance'})
	)

	df_both_left = _left_filter(df_both_left, df_close_merge, merge_cols)
	df_both_right = _left_filter(df_both_right, df_close_merge, merge_cols)

	assert pd.merge(
		df_both_left, df_both_right, how='inner', on=merge_cols
	).empty, 'Check merging steps, df_both_left and df_both_right should be disjoint'

	df_final_merge = pd.concat([df_close_merge, df_both_left, df_both_right], ignore_index=True)

	assert len(df_both_far_left) + len(df_close_merge) + len(df_both_left) == (
		len(df_left) - len(only_left)
	), 'Check merging steps, validation failed!'

	assert len(df_both_far_right) + len(df_close_merge) + len(df_both_right) == (
		len(df_right) - len(only_right)
	), 'Check merging steps, validation failed!'

	original_cols = df_both.columns.to_list()
	df_both = (
		pd.merge(df_both, df_final_merge, how='left', on=filter_left_cols, indicator=True)
		.loc[lambda x: x['_merge'] == 'left_only']
		.iloc[:, range(len(original_cols))]
	)
	df_both.columns = original_cols

	df_both = (
		pd.merge(df_both, df_final_merge, how='left', on=filter_right_cols, indicator=True)
		.loc[lambda x: x['_merge'] == 'left_only']
		.iloc[:, range(len(original_cols))]
	)
	df_both.columns = original_cols

	assert (df_both.Distance > MAX_DIST).all(), 'Check merging steps, validation failed!'

	assert (
		pd.merge(
			df_both,
			df_both_far_left,
			how='left',
			on=filter_left_cols,
			indicator=True,
			copy=False,
		)
		.loc[lambda x: x['_merge'] == 'left_only']
		.iloc[:, range(len(original_cols))]
		.empty
	), 'Check merging steps, validation failed!'

	assert (
		pd.merge(
			df_both,
			df_both_far_right,
			how='left',
			on=filter_right_cols,
			indicator=True,
			copy=False,
		)
		.loc[lambda x: x['_merge'] == 'left_only']
		.iloc[:, range(len(original_cols))]
		.empty
	), 'Check merging steps, validation failed!'

	for col in cols2merge:
		df_final_merge[f'{col}{left_suffix}'] = (
			df_final_merge[f'{col}{left_suffix}'] + ' | ' + df_final_merge[f'{col}{right_suffix}']
		)

	df_final_merge = df_final_merge[left_cols]
	df_final_merge.columns = only_left.columns

	df_both_far_left = df_both_far_left[left_cols]
	df_both_far_left.columns = only_left.columns

	df_both_far_right = df_both_far_right[right_cols]
	df_both_far_right.columns = only_right.columns

	return pd.concat(
		[only_left, df_both_far_left, df_final_merge, only_right, df_both_far_right],
		ignore_index=True,
	).astype('string', copy=False)


def _left_filter(df, df_close_merge, merge_cols):
	df = pd.merge(df, df_close_merge, how='left', on=merge_cols, indicator=True, copy=False)
	df = df.loc[df['_merge'] == 'left_only']
	return df.drop(['_merge', 'Distance_y'], axis=1).rename(columns={'Distance_x': 'Distance'})

## Funções Auxiliares

In [ ]:
# |export
def cast2float(column: pd.Series) -> pd.Series:
	return pd.to_numeric(
		column,
		downcast='float',
		errors='coerce',
		dtype_backend='numpy_nullable',
	).fillna(-1.0)


def cast2int(column: pd.Series) -> pd.Series:
	return pd.to_numeric(
		column,
		downcast='integer',
		errors='coerce',
		dtype_backend='numpy_nullable',
	).fillna(-1)


def cast2str(column: pd.Series) -> pd.Series:
	column.replace('', '-1', inplace=True)
	return column.astype('string', copy=False).fillna('-1')


def cast2cat(column: pd.Series) -> pd.Series:
	column.replace('', '-1', inplace=True)
	return column.fillna('-1').astype('category', copy=False)


def format_types(df):
	df['Frequência'] = df['Frequência'].astype('float')
	for col in FLOAT_COLUMNS:
		df[col] = cast2float(df[col])
	for col in INT_COLUMNS:
		df[col] = cast2int(df[col])
	for col in CAT_COLUMNS:
		df[col] = cast2cat(df[col])
	for col in STR_COLUMNS:
		df[col] = cast2str(df[col])
	return df


def merge_dicts(json_log: str):
	"""
	Mescla dicionários em uma lista com base no valor da chave "Processamento".

	Argumentos:
	  list_of_dicts: Uma lista contendo dicionários com as mesmas chaves.

	Retorno:
	  Um dicionário com os dicionários mesclados.
	"""
	from collections import Counter

	if not (list_of_dicts := [i for i in json.loads(json_log) if i != '[]']):
		return '[]'
	merged_dict = defaultdict(list)
	for item in list_of_dicts:
		processamento = item['Processamento']
		merged_dict[processamento].append(item)

	for key, value in merged_dict.items():
		if len(value) == 1:
			merged_dict[key] = value[0]
		else:
			merged_dict[key] = _merge_sub_dicts(value)

	output_list = []

	for log_dict in merged_dict.values():
		if log_dict.get('Coluna') == '#Estação':
			if value := log_dict.get('Original'):
				value = eval(value)
				if isinstance(value, list):
					log_dict['Original'] = str(sorted(list(set(value))))
				output_list.append(log_dict)

	return json.dumps(output_list, ensure_ascii=False)


def _merge_sub_dicts(list_of_sub_dicts):
	"""
	Mescla dicionários que possuem a mesma chave "Processamento".

	Argumentos:
	  list_of_sub_dicts: Uma lista contendo dicionários com a mesma chave "Processamento".

	Retorno:
	  Um dicionário com os dicionários mesclados.
	"""
	merged_dict = {}
	for key in list_of_sub_dicts[0].keys():
		if key == 'Processamento':
			merged_dict[key] = list_of_sub_dicts[0][key]
		else:
			merged_dict[key] = [sub_dict[key] for sub_dict in list_of_sub_dicts]

	return merged_dict
